# 02 - Validation of the loaded data

Runs every query in `sql/validation/`, compares the Parquet lake with PostgreSQL, and saves the results to
`docs/validation_results/` (a CSV for the report and a Markdown summary).

Run it **after** `parse_games.py`, `load_postgres.py` and `sql/generate_derived.sql` (and again after the labels are loaded).
It contains no usernames, so its outputs are safe to commit.

Check levels: **hard** = must pass (a failure stops the pipeline), **soft** = warning only, **info** = numbers for the report.

In [1]:
import datetime as dt
import sys
from pathlib import Path

import pandas as pd
import psycopg2
import pyarrow.dataset as ds

# Repo root: this notebook lives in notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.common.config import load_config

cfg = load_config("config/local.yaml")
months = cfg["months"]
pg = cfg["postgres"]
conn = psycopg2.connect(host=pg["host"], port=pg["port"], dbname=pg["dbname"],
                        user=pg["user"], password=pg["password"] or None)
conn.autocommit = True  # the validation queries only read
print("Connected to", pg["dbname"], "| months:", months)

Connected to chess | months: ['2026-09']


In [2]:
def run_sql_file(path):
    """Run one validation .sql file and return its result as a DataFrame."""
    with conn.cursor() as cur:
        cur.execute(Path(path).read_text(encoding="utf-8"))
        columns = [c.name for c in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=columns)


def pg_scalar(query, params=None):
    with conn.cursor() as cur:
        cur.execute(query, params)
        return cur.fetchone()[0]

## 1. Run the SQL validation files

In [3]:
results = []
for sql_file in sorted((ROOT / "sql" / "validation").glob("*.sql")):
    df = run_sql_file(sql_file)
    df.insert(0, "file", sql_file.name)
    results.append(df)
    print(sql_file.name)
    display(df.drop(columns="file"))

validation = pd.concat(results, ignore_index=True)
for col in ("observed", "max_allowed"):
    validation[col] = pd.to_numeric(validation[col], errors="coerce")

01_row_counts.sql


,check_name,level,observed,max_allowed,status
0,core.games rows (compare with Parquet games),info,124913,None,INFO
1,core.moves rows (compare with Parquet moves),info,8302882,None,INFO
2,restricted.identity rows (compare with Parquet...,info,69392,None,INFO
3,core.players rows,info,69392,None,INFO
4,core.baselines rows,info,55,None,INFO
5,core.player_labels rows,info,69392,None,INFO
6,ref.rating_bands rows,info,12,None,INFO
7,ref.time_classes rows,info,6,None,INFO
8,core.players vs restricted.identity: row diffe...,hard,0,0,PASS
9,core.players vs distinct players in games: row...,hard,0,0,PASS


02_keys.sql


,check_name,level,observed,max_allowed,status
0,games: duplicate game_id,hard,0,0,PASS
1,games: null game_id,hard,0,0,PASS
2,"moves: duplicate (game_id, ply)",hard,0,0,PASS
3,identity: duplicate player_id,hard,0,0,PASS
4,identity: duplicate username,hard,0,0,PASS
5,moves: game_id missing from games (orphans),hard,0,0,PASS
6,games: white_id = black_id,hard,0,0,PASS
7,games: player missing from restricted.identity,hard,0,0,PASS
8,moves: player_id does not match the game's pla...,hard,0,0,PASS
9,moves: colour does not match ply parity (odd p...,hard,0,0,PASS


03_n_plies.sql


,check_name,level,observed,max_allowed,status
0,games: n_plies differs from the number of move...,hard,0,0,PASS
1,games: ply numbers not contiguous (max ply <> ...,hard,0,0,PASS
2,games: n_plies = 0 (no moves),soft,303,0,WARN
3,games checked,info,124913,None,INFO
4,total plies in games.n_plies,info,8302882,None,INFO
5,total rows in core.moves,info,8302882,None,INFO


... (2 outputs left out)

05_label_distribution.sql


,check_name,level,observed,max_allowed,status
0,player_labels: table is empty (labels not load...,soft,0,0,PASS
1,label = positive (players),info,566,None,INFO
2,label = negative (players),info,68128,None,INFO
3,label = unknown (players),info,698,None,INFO
4,label = positive (percent of labelled players),info,0.816,None,INFO
5,found = false (players the API did not return),info,17,None,INFO
6,player_labels: label inconsistent with the rul...,hard,0,0,PASS


06_label_coverage.sql


,check_name,level,observed,max_allowed,status
0,players without a label in core.player_labels,soft,0,0,PASS
1,labels for players that are not in core.players,soft,0,0,PASS
2,players with a label (percent of core.players),info,100.00,None,INFO


## 2. Parquet lake vs PostgreSQL row counts

SQL cannot read Parquet, so this comparison lives in the notebook. A difference means rows were lost or duplicated in the load.

In [4]:
lake = Path(cfg["paths"]["lake_dir"])


def parquet_count(sub_path, month_column):
    """Rows of a Parquet dataset for the configured months."""
    dataset = ds.dataset(str(lake / sub_path), format="parquet", partitioning="hive")
    return dataset.count_rows(filter=ds.field(month_column).isin(months))


pairs = {
    "games": (parquet_count("games", "source_month"),
              pg_scalar("SELECT count(*) FROM core.games WHERE source_month = ANY(%s)", (months,))),
    "moves": (parquet_count("moves", "source_month"),
              pg_scalar("SELECT count(*) FROM core.moves m JOIN core.games g USING (game_id) "
                        "WHERE g.source_month = ANY(%s)", (months,))),
    "identity": (parquet_count("restricted/identity", "first_seen_month"),
                 pg_scalar("SELECT count(*) FROM restricted.identity WHERE first_seen_month = ANY(%s)", (months,))),
}
comparison = pd.DataFrame(
    [(name, p, q, p == q) for name, (p, q) in pairs.items()],
    columns=["table", "parquet_rows", "postgres_rows", "match"],
)
display(comparison)

quarantine_rows = parquet_count("quarantine", "source_month")
print("Games rejected by the parser (quarantine):", quarantine_rows)

extra = pd.DataFrame({
    "file": "notebook",
    "check_name": [f"parquet vs postgresql: {t} row difference" for t in comparison["table"]]
                  + ["parquet quarantine rows (games rejected by the parser)"],
    "level": ["hard"] * len(comparison) + ["info"],
    "observed": [abs(p - q) for p, q in zip(comparison["parquet_rows"], comparison["postgres_rows"])] + [quarantine_rows],
    "max_allowed": [0] * len(comparison) + [None],
})
extra["status"] = [
    "INFO" if lvl == "info" else ("PASS" if obs <= mx else "FAIL")
    for lvl, obs, mx in zip(extra["level"], extra["observed"], extra["max_allowed"])
]
validation = pd.concat([validation, extra], ignore_index=True)

,table,parquet_rows,postgres_rows,match
0,games,124913,124913,True
1,moves,8302882,8302882,True
2,identity,69392,69392,True


Games rejected by the parser (quarantine): 1


## 3. Summary and save

In [5]:
run_at = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
validation["run_at_utc"] = run_at

hard_failures = validation[(validation["level"] == "hard") & (validation["status"] == "FAIL")]
summary = validation.groupby(["level", "status"]).size().unstack(fill_value=0)
display(summary)

out_dir = ROOT / "docs" / "validation_results"
out_dir.mkdir(parents=True, exist_ok=True)
validation.to_csv(out_dir / "validation_results.csv", index=False)

lines = [
    "# Validation results", "",
    f"Run at: {run_at}", f"Months: {', '.join(months)}",
    f"Hard checks failed: {len(hard_failures)}", "",
    "| File | Check | Level | Observed | Max allowed | Status |",
    "|---|---|---|---|---|---|",
]
def fmt(x):
    """Exact number text: whole numbers without decimals, others rounded to 4 decimals, never scientific notation."""
    if pd.isna(x):
        return ""
    if float(x).is_integer():
        return str(int(x))
    return format(round(float(x), 4), "f").rstrip("0").rstrip(".")


for r in validation.itertuples():
    lines.append(f"| {r.file} | {r.check_name} | {r.level} | {fmt(r.observed)} | {fmt(r.max_allowed)} | {r.status} |")
(out_dir / "validation_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Saved:", out_dir / "validation_results.csv", "and", out_dir / "validation_summary.md")

status,INFO,PASS,WARN
level,,,
hard,0,26,0
info,26,0,0
soft,0,5,2


Saved: /Users/santiagolandinez/Documents/chess-cheat-detector/docs/validation_results/validation_results.csv and /Users/santiagolandinez/Documents/chess-cheat-detector/docs/validation_results/validation_summary.md


In [6]:
conn.close()
assert hard_failures.empty, f"Hard validation checks failed:\n{hard_failures}"
print("All hard checks passed.")

All hard checks passed.
